# Autonomous Trailer: A Knowledge-Based Agent
**Unit:** Artificial Intelligence

| Group member | Registration no. |
|---|---|
| Maina Jeremy Gachanga | 190280 |
| Ng'ang'a Austin Thayu | 181807 |
| Muiya Adrian Kioko | 178619 |

This notebook develops the self-driving trailer from practical exercise 3 as a **knowledge-based agent**. The trailer must travel from **Nairobi to Kisumu** on Kenya's highway network. It stores what it knows about the road network in a **knowledge base (KB)**, and it answers questions by querying that KB.

**Work split**
- **Steps 1-5** (task, knowledge sources, vocabulary, building the KB, adding new facts): Ng'ang'a Austin Thayu
- **Steps 6-8** (problem instance, reasoning engine, user interaction, queries): Muiya Adrian Kioko
- **Steps 9-10** (debugging and final reflection): Maina Jeremy Gachanga

**Step 1: Identify the Task**

The task is to help a self-driving trailer decide **which route to take from Nairobi to Kisumu**, using what it knows about the roads.

**Goal:** Deliver the cargo safely at the **lowest total cost**. Cost is driving distance in km, plus a penalty for hazards on the road. Closed roads must never be used.

**What the agent must be able to do**
- Answer questions about towns and roads, such as "is this road open?" and "what does it cost?"
- Take in **new facts** when the world changes, for example a closed road or a new hazard
- Give correct answers after the knowledge has changed

**Step 2: Assemble Relevant Knowledge**

The agent needs to know:
- Which towns exist and which county each is in
- Which towns are connected by a highway, and how far apart they are
- Whether each road is currently open or closed
- Which hazards are on each road, and how much each hazard should cost us
- The general rules for using this information

**Sources of knowledge**
| Knowledge | Source |
|---|---|
| Towns, 21 directed roads and distances | Directed graph and distance table from **practical exercise 3** (`design.md`, `UCS_Trailer.ipynb`). Distances are Google Maps driving distances rounded to the nearest km. |
| Hazard types (potholes, fog, livestock, escarpments, etc.) | The environment description in our **PEAS table** (practical exercise 3). |
| Hazard penalties | Our own design choice (explained in Step 4). |

**APIs and web scraping:** *None were used.* The knowledge base is **hard-coded** from our practical exercise 3 data, so no extra code needs to be uploaded to the repository.

**Step 3: Define the Vocabulary**

In Python, **predicates** are stored as keys in dictionaries and as rules (functions). **Constants** are the fixed values.

| Predicate | Meaning | Where it lives in the KB |
|---|---|---|
| `Town(x)` | x is a town on the network | `kb["towns"]` |
| `County(x, c)` | town x is in county c | `kb["towns"][x]["county"]` |
| `Role(x, r)` | town x is the start, goal, dead end or intermediate | `kb["towns"][x]["role"]` |
| `Road(a, b, km)` | a directed road from a to b of length km | `kb["roads"][a][b]` |
| `Status(a, b, s)` | road a to b is open or closed | `kb["status"][(a, b)]` |
| `Hazard(a, b, h)` | hazard h is present on road a to b | `kb["hazards"][(a, b)]` |
| `HazardPenalty(h, p)` | hazard h adds p km-equivalent of cost | `kb["rules"]["hazard_penalty_km"][h]` |
| `Usable(a, b)` | road exists **and** is open (a derived rule) | function `is_reachable_road(a, b)` |
| `EffectiveCost(a, b, c)` | km + hazard penalties (a derived rule) | function `get_neighbors(a)` |

**Constants**
- **Towns (19):** Nairobi, Ruiru, Athi River, Naivasha, Narok, Thika, Karatina, Nyeri, Nyahururu, Nakuru, Makindu, Kajiado, Gilgil, Kericho, Kisumu, Bomet, Kisii, Migori, Homa Bay
- **Start:** Nairobi | **Goal:** Kisumu
- **Status values:** open, closed
- **Roles:** start, goal, dead_end, intermediate

**Step 4: Encode General Knowledge about the Domain** `[3 marks]`

The knowledge base is a single Python dictionary called `kb` with five sections: `towns`, `roads`, `status`, `hazards` and `rules`. The cells below create the raw data, build `kb` from it, add query helpers, and then check that the KB is correct.

In [1]:
# ------------------------------------------------------------
# STEP 4a: RAW DATA (taken from practical exercise 3)
# ------------------------------------------------------------

# TOWNS: every town on the network (19 in total).
# Each entry is  town -> (county, role)
# "role" says what the town means for our problem:
#   "start"        -> where the trailer begins
#   "goal"         -> where the trailer must arrive
#   "dead_end"     -> a town with no onward road in our graph (cannot lead to Kisumu)
#   "intermediate" -> any other town on the way
TOWNS = {
    "Nairobi":    ("Nairobi",   "start"),
    "Ruiru":      ("Kiambu",    "intermediate"),
    "Athi River": ("Machakos",  "intermediate"),
    "Naivasha":   ("Nakuru",    "intermediate"),
    "Narok":      ("Narok",     "intermediate"),
    "Thika":      ("Kiambu",    "intermediate"),
    "Karatina":   ("Nyeri",     "intermediate"),
    "Nyeri":      ("Nyeri",     "intermediate"),
    "Nyahururu":  ("Laikipia",  "intermediate"),
    "Nakuru":     ("Nakuru",    "intermediate"),
    "Makindu":    ("Makueni",   "dead_end"),
    "Kajiado":    ("Kajiado",   "dead_end"),
    "Gilgil":     ("Nakuru",    "intermediate"),
    "Kericho":    ("Kericho",   "intermediate"),
    "Kisumu":     ("Kisumu",    "goal"),
    "Bomet":      ("Bomet",     "intermediate"),
    "Kisii":      ("Kisii",     "intermediate"),
    "Migori":     ("Migori",    "dead_end"),
    "Homa Bay":   ("Homa Bay",  "intermediate"),
}

# EDGES: the 21 directed roads, as (from, to, distance in km).
# These are exactly the 21 rows of the distance table in practical exercise 3.
EDGES = [
    ("Nairobi",    "Ruiru",       25),
    ("Nairobi",    "Athi River",  30),
    ("Nairobi",    "Naivasha",    91),
    ("Nairobi",    "Narok",      143),
    ("Ruiru",      "Thika",       21),
    ("Thika",      "Karatina",    88),
    ("Karatina",   "Nyeri",       25),
    ("Nyeri",      "Nyahururu",  104),
    ("Nyahururu",  "Nakuru",      68),
    ("Athi River", "Makindu",    149),
    ("Athi River", "Kajiado",     50),
    ("Naivasha",   "Gilgil",      30),
    ("Gilgil",     "Nakuru",      42),
    ("Nakuru",     "Kericho",    107),
    ("Kericho",    "Kisumu",      84),
    ("Narok",      "Bomet",       81),
    ("Bomet",      "Kisii",       82),
    ("Kisii",      "Kisumu",     114),
    ("Kisii",      "Migori",      67),
    ("Kisii",      "Homa Bay",    52),
    ("Homa Bay",   "Kisumu",     108),
]

print(f"Raw data loaded: {len(TOWNS)} towns, {len(EDGES)} roads")

Raw data loaded: 19 towns, 21 roads


In [2]:
# ------------------------------------------------------------
# STEP 4b: BUILD THE KNOWLEDGE BASE
# ------------------------------------------------------------
# The KB is ONE dictionary with five sections:
#   "towns"   : town -> {"county": ..., "role": ...}
#   "roads"   : from_town -> {to_town: distance_km}     (directed roads)
#   "status"  : (from_town, to_town) -> "open" or "closed"
#   "hazards" : (from_town, to_town) -> [list of hazard names]  (empty at the start)
#   "rules"   : general rules that apply to the whole domain

kb = {
    "towns":   {},
    "roads":   {},
    "status":  {},
    "hazards": {},      # no hazards are known yet; we add them in Step 5

    "rules": {
        # The only two values a road's status may take.
        "valid_status": ["open", "closed"],

        # RULE: each hazard adds a penalty to the cost of a road.
        # Penalties are in "km-equivalent": a road with fog costs
        # its distance PLUS 10. These numbers are our own design choice;
        # a higher number means the hazard is more dangerous/slow.
        # The hazard types come from the environment in our PEAS table.
        "hazard_penalty_km": {
            "pothole":            5,
            "fog":               10,
            "livestock crossing": 10,
            "steep escarpment":  10,
            "heavy traffic":     15,
            "roadworks":         20,
            "flooding":          40,
        },

        # RULES written in words, to document how the helper functions behave.
        "usable_road":    "A road can be used only if it exists AND its status is 'open'.",
        "effective_cost": "Effective cost of a road = distance in km + the penalties of all hazards on it.",
        "dead_end":       "A dead-end town has no outgoing roads, so it can never lead on to the goal.",
    },
}

# Fill in the towns. Every town also gets an (empty) road dictionary,
# so that dead-end towns still exist as keys in kb["roads"].
for town, (county, role) in TOWNS.items():
    kb["towns"][town] = {"county": county, "role": role}
    kb["roads"][town] = {}

# Fill in the roads. Every road starts out "open".
for from_town, to_town, km in EDGES:
    kb["roads"][from_town][to_town] = km
    kb["status"][(from_town, to_town)] = "open"

print("Knowledge base built.")
print("Sections:", list(kb.keys()))

Knowledge base built.
Sections: ['towns', 'roads', 'status', 'hazards', 'rules']


In [3]:
# ------------------------------------------------------------
# STEP 4c: HELPER FUNCTIONS 
# These only READ the KB. They never change it.
# ------------------------------------------------------------

def normalise(name):
    """Clean up a town name so 'nairobi ' and 'NAIROBI' both become 'Nairobi'."""
    return name.strip().title()


def town_exists(town):
    """True if the town is in the knowledge base."""
    return normalise(town) in kb["towns"]


def road_exists(a, b):
    """True if there is a directed road from town a to town b (open OR closed)."""
    a, b = normalise(a), normalise(b)
    return b in kb["roads"].get(a, {})


def is_reachable_road(a, b):
    """
    RULE: Usable(a, b)
    True only if the road exists AND it is currently open.
    """
    a, b = normalise(a), normalise(b)
    return road_exists(a, b) and kb["status"].get((a, b)) == "open"


def hazard_penalty(a, b):
    """Total penalty (in km-equivalent) of all hazards currently on the road a -> b."""
    a, b = normalise(a), normalise(b)
    penalties = kb["rules"]["hazard_penalty_km"]
    hazards_on_road = kb["hazards"].get((a, b), [])   # [] if there are none
    return sum(penalties[h] for h in hazards_on_road)


def get_neighbors(town):
    """
    The MAIN function for Person B's search.
    Returns a list of (next_town, effective_cost) for every OPEN road leaving `town`.
        effective_cost = distance_km + hazard_penalty
    - Closed roads are left out.
    - Dead-end or unknown towns return an empty list [].
    """
    town = normalise(town)
    neighbors = []
    for next_town, km in kb["roads"].get(town, {}).items():
        if is_reachable_road(town, next_town):
            neighbors.append((next_town, km + hazard_penalty(town, next_town)))
    return neighbors


def route_cost(route):
    """
    Total effective cost of a route given as a list of towns,
    e.g. ["Nairobi", "Naivasha", "Gilgil"].
    Returns None if any leg of the route is missing or closed.
    """
    total = 0
    for a, b in zip(route, route[1:]):                # look at each pair of neighbours
        costs_from_a = dict(get_neighbors(a))         # {next_town: cost}
        if normalise(b) not in costs_from_a:
            return None                               # this leg cannot be driven
        total += costs_from_a[normalise(b)]
    return total


def show_kb():
    """Print every road in the KB with its distance, status and hazards."""
    print("\n--- Current Knowledge Base ---")
    for town, roads_out in kb["roads"].items():
        if not roads_out:
            print(f"{town:11} -> dead end (no outgoing roads)")
            continue
        for next_town, km in roads_out.items():
            status = kb["status"][(town, next_town)]
            hazards = kb["hazards"].get((town, next_town), [])
            hazard_text = f" | hazards: {', '.join(hazards)}" if hazards else ""
            print(f"{town:11} -> {next_town:11} {km:4} km  [{status}]{hazard_text}")
    print("------------------------------")

In [4]:
# ------------------------------------------------------------
# STEP 4d: VIEW THE KB AND CHECK IT IS CORRECT
# ------------------------------------------------------------
show_kb()

# --- Sanity checks: if any of these fail, Python shows an AssertionError ---
total_roads = sum(len(r) for r in kb["roads"].values())
assert len(kb["towns"]) == 19, "Expected 19 towns"
assert total_roads == 21, "Expected 21 roads"

# Every road must connect two towns that exist.
for a, roads_out in kb["roads"].items():
    for b in roads_out:
        assert a in kb["towns"] and b in kb["towns"], f"Unknown town in road {a} -> {b}"

# Dead-end towns must have no outgoing roads.
for town, info in kb["towns"].items():
    if info["role"] == "dead_end":
        assert kb["roads"][town] == {}, f"{town} is a dead end but has roads"

# The best route found in practical exercise 3 must still cost 354 km.
KNOWN_BEST_ROUTE = ["Nairobi", "Naivasha", "Gilgil", "Nakuru", "Kericho", "Kisumu"]
assert route_cost(KNOWN_BEST_ROUTE) == 354, "Route cost does not match practical exercise 3"

print(f"\nAll checks passed: {len(kb['towns'])} towns, {total_roads} roads.")
print("Cost of the known best route:", route_cost(KNOWN_BEST_ROUTE), "km")


--- Current Knowledge Base ---
Nairobi     -> Ruiru         25 km  [open]
Nairobi     -> Athi River    30 km  [open]
Nairobi     -> Naivasha      91 km  [open]
Nairobi     -> Narok        143 km  [open]
Ruiru       -> Thika         21 km  [open]
Athi River  -> Makindu      149 km  [open]
Athi River  -> Kajiado       50 km  [open]
Naivasha    -> Gilgil        30 km  [open]
Narok       -> Bomet         81 km  [open]
Thika       -> Karatina      88 km  [open]
Karatina    -> Nyeri         25 km  [open]
Nyeri       -> Nyahururu    104 km  [open]
Nyahururu   -> Nakuru        68 km  [open]
Nakuru      -> Kericho      107 km  [open]
Makindu     -> dead end (no outgoing roads)
Kajiado     -> dead end (no outgoing roads)
Gilgil      -> Nakuru        42 km  [open]
Kericho     -> Kisumu        84 km  [open]
Kisumu      -> dead end (no outgoing roads)
Bomet       -> Kisii         82 km  [open]
Kisii       -> Kisumu       114 km  [open]
Kisii       -> Migori        67 km  [open]
Kisii       -> Homa

**Step 5: Add New Facts to the Knowledge Base** `[4 marks]`

A knowledge-based agent must be able to **learn new facts** while it is running. We provide functions to change each part of the KB:

| Function | What it does |
|---|---|
| `add_town(name, county)` | Adds a new town |
| `remove_town(name)` | Removes a town and every road into or out of it |
| `add_road(a, b, km)` | Adds a new directed road (starts open) |
| `remove_road(a, b)` | Deletes a road |
| `update_distance(a, b, km)` | Corrects a distance |
| `set_status(a, b, status)` | Opens or closes a road |
| `add_hazard_type(name, penalty)` | Teaches the agent a new kind of hazard (a new **rule**) |
| `add_hazard(a, b, hazard)` | Records a hazard on a road |
| `clear_hazards(a, b)` | Removes all hazards from a road |

Each function checks its input first, so a bad fact cannot corrupt the KB. Each prints a message and returns `True` (success) or `False` (rejected).

In [5]:
# ------------------------------------------------------------
# STEP 5a: ADDING / REMOVING TOWNS AND ROADS
# ------------------------------------------------------------

def add_town(name, county, role="intermediate"):
    """Add a new town to the KB. It starts with no roads."""
    name = normalise(name)
    if town_exists(name):
        print(f"Rejected: town '{name}' already exists.")
        return False
    kb["towns"][name] = {"county": county, "role": role}
    kb["roads"][name] = {}                 # a new town starts with no outgoing roads
    print(f"Added town: {name} ({county} County)")
    return True


def add_road(a, b, km):
    """Add a new directed road a -> b. It starts out 'open'."""
    a, b = normalise(a), normalise(b)
    if not (town_exists(a) and town_exists(b)):
        print(f"Rejected: both '{a}' and '{b}' must already be towns in the KB (use add_town first).")
        return False
    if road_exists(a, b):
        print(f"Rejected: road {a} -> {b} already exists (use update_distance to change it).")
        return False
    if not isinstance(km, (int, float)) or km <= 0:
        print("Rejected: distance must be a positive number.")
        return False
    kb["roads"][a][b] = km
    kb["status"][(a, b)] = "open"
    print(f"Added road: {a} -> {b} ({km} km, open)")
    return True


def remove_road(a, b, verbose=True):
    """Delete the road a -> b, together with its status and hazards."""
    a, b = normalise(a), normalise(b)
    if not road_exists(a, b):
        if verbose:
            print(f"Rejected: no road {a} -> {b} to remove.")
        return False
    del kb["roads"][a][b]
    kb["status"].pop((a, b), None)         # pop(..., None) = delete if present, no error if not
    kb["hazards"].pop((a, b), None)
    if verbose:
        print(f"Removed road: {a} -> {b}")
    return True


def remove_town(name):
    """Remove a town and every road going into or out of it."""
    name = normalise(name)
    if not town_exists(name):
        print(f"Rejected: town '{name}' does not exist.")
        return False
    if kb["towns"][name]["role"] in ("start", "goal"):
        print(f"Rejected: '{name}' is the start or goal town and cannot be removed.")
        return False
    for other in list(kb["roads"]):                       # roads INTO the town
        remove_road(other, name, verbose=False)
    for next_town in list(kb["roads"][name]):             # roads OUT of the town
        remove_road(name, next_town, verbose=False)
    del kb["roads"][name]
    del kb["towns"][name]
    print(f"Removed town: {name} (and all its roads)")
    return True


def update_distance(a, b, km):
    """Correct the distance of an existing road."""
    a, b = normalise(a), normalise(b)
    if not road_exists(a, b):
        print(f"Rejected: no road {a} -> {b} to update.")
        return False
    if not isinstance(km, (int, float)) or km <= 0:
        print("Rejected: distance must be a positive number.")
        return False
    old = kb["roads"][a][b]
    kb["roads"][a][b] = km
    print(f"Updated distance {a} -> {b}: {old} km -> {km} km")
    return True

In [6]:
# ------------------------------------------------------------
# STEP 5b: ROAD STATUS AND HAZARDS
# ------------------------------------------------------------

def set_status(a, b, status):
    """Open or close a road. Only the values in kb['rules']['valid_status'] are allowed."""
    a, b = normalise(a), normalise(b)
    status = status.strip().lower()
    if not road_exists(a, b):
        print(f"Rejected: no road {a} -> {b}.")
        return False
    if status not in kb["rules"]["valid_status"]:
        print(f"Rejected: status must be one of {kb['rules']['valid_status']}.")
        return False
    kb["status"][(a, b)] = status
    print(f"Road {a} -> {b} is now {status.upper()}")
    return True


def add_hazard_type(name, penalty_km):
    """Teach the agent a NEW kind of hazard (this adds a new rule to kb['rules'])."""
    name = name.strip().lower()
    if name in kb["rules"]["hazard_penalty_km"]:
        print(f"Rejected: hazard type '{name}' is already known.")
        return False
    if not isinstance(penalty_km, (int, float)) or penalty_km < 0:
        print("Rejected: penalty must be a number that is 0 or more.")
        return False
    kb["rules"]["hazard_penalty_km"][name] = penalty_km
    print(f"Added hazard type: '{name}' (penalty {penalty_km} km)")
    return True


def add_hazard(a, b, hazard):
    """Record that a known hazard is present on the road a -> b."""
    a, b = normalise(a), normalise(b)
    hazard = hazard.strip().lower()
    if not road_exists(a, b):
        print(f"Rejected: no road {a} -> {b}.")
        return False
    if hazard not in kb["rules"]["hazard_penalty_km"]:
        print(f"Rejected: unknown hazard '{hazard}'. Known: {list(kb['rules']['hazard_penalty_km'])}")
        return False
    current = kb["hazards"].setdefault((a, b), [])        # create an empty list if none yet
    if hazard in current:
        print(f"Rejected: '{hazard}' is already recorded on {a} -> {b}.")
        return False
    current.append(hazard)
    print(f"Added hazard '{hazard}' on {a} -> {b}")
    return True


def clear_hazards(a, b):
    """Remove all hazards from the road a -> b."""
    a, b = normalise(a), normalise(b)
    if not road_exists(a, b):
        print(f"Rejected: no road {a} -> {b}.")
        return False
    kb["hazards"].pop((a, b), None)
    print(f"Cleared all hazards on {a} -> {b}")
    return True

In [7]:
# ------------------------------------------------------------
# STEP 5c: USER-FRIENDLY VERSION (asks the user to type the new fact)
# ------------------------------------------------------------
# This is defined but NOT called, because input() pauses the notebook and
# would stop "Run all" from finishing. To try it, remove the # on the last line.

def add_road_interactive():
    """Ask the user for a new road and add it to the KB."""
    a  = input("From which town? ")
    b  = input("To which town? ")
    try:
        km = float(input("Distance in km? "))
    except ValueError:
        print("Rejected: distance must be a number.")
        return False
    return add_road(a, b, km)

# add_road_interactive()

In [8]:
# ------------------------------------------------------------
# STEP 5d: DEMONSTRATION - add new facts and see how answers change
# At the end we RESTORE the KB so the next person (Person B) starts from the clean original.
# ------------------------------------------------------------

print("=== 5a. NEW TOWN AND ROAD ===")
print("Roads out of Nakuru BEFORE:", get_neighbors("Nakuru"))
add_town("Eldoret", "Uasin Gishu")
add_road("Nakuru", "Eldoret", 155)            # illustrative distance
print("Roads out of Nakuru AFTER: ", get_neighbors("Nakuru"))

print("\n=== 5b. CLOSE A ROAD ===")
print("Cost of best known route BEFORE:", route_cost(KNOWN_BEST_ROUTE), "km")
set_status("Naivasha", "Gilgil", "closed")
print("Roads out of Naivasha now:      ", get_neighbors("Naivasha"))
print("Cost of best known route AFTER: ", route_cost(KNOWN_BEST_ROUTE), "(None = route blocked)")
set_status("Naivasha", "Gilgil", "open")      # reopen it

print("\n=== 5c. NEW HAZARD ===")
add_hazard("Nakuru", "Kericho", "fog")
print("Roads out of Nakuru with fog:", get_neighbors("Nakuru"))
print("Cost of best known route with fog:", route_cost(KNOWN_BEST_ROUTE), "km (354 + 10)")

print("\n=== 5d. NEW HAZARD TYPE (a new rule) ===")
add_hazard_type("landslide", 30)              # temporary: removed again in the restore step below
add_hazard("Narok", "Bomet", "landslide")
print("Roads out of Narok:", get_neighbors("Narok"), "(81 + 30)")

print("\n=== 5e. REJECTED (INVALID) FACTS ===")
add_road("Nairobi", "Mombasa", 480)           # Mombasa is not in the KB
set_status("Nairobi", "Ruiru", "maybe")       # not a valid status
add_hazard("Nairobi", "Ruiru", "tornado")     # unknown hazard type
remove_town("Kisumu")                         # goal town is protected

# --- RESTORE the original KB so the demo leaves no side effects ---
print("\n=== RESTORING THE ORIGINAL KB ===")
remove_town("Eldoret")                        # also removes the Nakuru -> Eldoret road
clear_hazards("Nakuru", "Kericho")            # remove the fog
clear_hazards("Narok", "Bomet")               # remove the landslide from the road
del kb["rules"]["hazard_penalty_km"]["landslide"]   # forget the landslide hazard type itself

# --- CHECK that everything is back to the original state ---
assert route_cost(KNOWN_BEST_ROUTE) == 354
assert sum(len(r) for r in kb["roads"].values()) == 21
assert len(kb["towns"]) == 19
assert len(kb["rules"]["hazard_penalty_km"]) == 7                 # original 7 hazard types
assert all(not h for h in kb["hazards"].values())                 # no hazards left on any road
assert all(s == "open" for s in kb["status"].values())            # every road is open again
print("KB restored: 19 towns, 21 roads, 7 hazard types, no hazards, best route costs 354 km.")

=== 5a. NEW TOWN AND ROAD ===
Roads out of Nakuru BEFORE: [('Kericho', 107)]
Added town: Eldoret (Uasin Gishu County)
Added road: Nakuru -> Eldoret (155 km, open)
Roads out of Nakuru AFTER:  [('Kericho', 107), ('Eldoret', 155)]

=== 5b. CLOSE A ROAD ===
Cost of best known route BEFORE: 354 km
Road Naivasha -> Gilgil is now CLOSED
Roads out of Naivasha now:       []
Cost of best known route AFTER:  None (None = route blocked)
Road Naivasha -> Gilgil is now OPEN

=== 5c. NEW HAZARD ===
Added hazard 'fog' on Nakuru -> Kericho
Roads out of Nakuru with fog: [('Kericho', 117), ('Eldoret', 155)]
Cost of best known route with fog: 364 km (354 + 10)

=== 5d. NEW HAZARD TYPE (a new rule) ===
Added hazard type: 'landslide' (penalty 30 km)
Added hazard 'landslide' on Narok -> Bomet
Roads out of Narok: [('Bomet', 111)] (81 + 30)

=== 5e. REJECTED (INVALID) FACTS ===
Rejected: both 'Nairobi' and 'Mombasa' must already be towns in the KB (use add_town first).
Rejected: status must be one of ['open', 

**Steps 6 & 7: Encode Specific Problem Instance (Facts) & Pose Queries and Get Answers** `[3 marks]`

Steps 4 and 5 stored **general** knowledge (the road network and the rules). Here we add the **specific facts of one trip**: the start, the goal, and today's road conditions. Then we build the **reasoning engine**: Uniform Cost Search (UCS) from practical 3, changed so that it **asks the KB** (`get_neighbors`) instead of reading a hard-coded graph.

| Query | Function |
|---|---|
| Best route? | `query_best_route(start, goal)` |
| Can I reach it at all? | `query_reachable(start, goal)` |
| What does it cost? | `query_cost(start, goal)` |
| What do we know about this road? | `query_road(a, b)` |
| Where can I get to from here? | `query_reachable_from(town)` |
| What is wrong on the roads today? | `query_active_problems()` |

User interaction is a **text menu** (`run_menu`).

In [9]:
# ------------------------------------------------------------
# STEP 6: ENCODE THE SPECIFIC PROBLEM INSTANCE (the facts of THIS trip)
# Step 4 stored GENERAL knowledge (the road network and the rules).
# Step 6 stores SPECIFIC facts: where we start, where we are going,
# and what the conditions are today.
# ------------------------------------------------------------
import copy

# --- 6a. Take a snapshot of the clean KB from Steps 4-5 ---------
# We will change the KB many times while testing. The snapshot lets us
# go back to a known state whenever we need to.
BASE_KB = copy.deepcopy(kb)

def restore_kb(snapshot):
    """Put the KB back to a saved snapshot. We change kb IN PLACE
    (clear + update) so every function that uses kb sees the change."""
    kb.clear()
    kb.update(copy.deepcopy(snapshot))

# --- 6b. The trip: start and goal -------------------------------
START, GOAL = "Nairobi", "Kisumu"

# --- 6c. Today's conditions, reported by the trailer's sensors --
# Each entry is (from_town, to_town, hazard). These are SPECIFIC facts,
# so we record them with the Step 5 function add_hazard().
TODAYS_CONDITIONS = [
    ("Nakuru",  "Kericho", "fog"),         # fog on the Mau escarpment
    ("Naivasha", "Gilgil",  "roadworks"),   # roadworks near Gilgil
    ("Narok",   "Bomet",   "flooding"),    # heavy rain in the south-west
]

print(f"Trip: {START} -> {GOAL}")
print("Recording today's conditions:")
for a, b, hazard in TODAYS_CONDITIONS:
    add_hazard(a, b, hazard)

# --- 6d. Snapshot "today" so we can return to it after each test -
TODAY_KB = copy.deepcopy(kb)
print("\nFacts about this trip are now in the KB.")

Trip: Nairobi -> Kisumu
Recording today's conditions:
Added hazard 'fog' on Nakuru -> Kericho
Added hazard 'roadworks' on Naivasha -> Gilgil
Added hazard 'flooding' on Narok -> Bomet

Facts about this trip are now in the KB.


In [10]:
# ------------------------------------------------------------
# STEP 7a: THE REASONING ENGINE - Uniform Cost Search over the KB
# This is the UCS from practical 3, with ONE change: instead of reading a
# hard-coded graph[town], it ASKS the KB with get_neighbors(town).
# So closed roads, hazards and new roads are all respected automatically.
# ------------------------------------------------------------
import heapq
from itertools import count

def ucs(start, goal, verbose=False):
    """
    Find the cheapest route from start to goal using only what the KB says.
    Returns (path, cost). If there is no route, or an input is invalid,
    returns (None, reason_text) so the caller can explain the problem.
    """
    raw_start, raw_goal = start, goal            # keep what the user typed for error messages
    start, goal = normalise(start), normalise(goal)

    # Guard clauses: refuse bad questions before searching
    if not town_exists(start):
        return None, f"I don't know a town called '{raw_start}'."
    if not town_exists(goal):
        return None, f"I don't know a town called '{raw_goal}'."
    if start == goal:
        return [start], 0                        # already there

    tie = count()                                # tie-breaker: first added is expanded first
    frontier = [(0, next(tie), start, [start])]  # entries: (cost so far, tie, town, path)
    explored = set()

    while frontier:
        g, _, town, path = heapq.heappop(frontier)   # take the CHEAPEST path
        if town in explored:
            continue                                 # a cheaper path to here was already used
        if verbose:
            print(f"  expand {' -> '.join(path)}  (g = {g})")
        if town == goal:                             # goal test on EXPANSION (as in practical 3)
            return path, g
        explored.add(town)

        for next_town, step_cost in get_neighbors(town):   # <-- the KB is queried here
            if next_town not in explored:
                heapq.heappush(frontier, (g + step_cost, next(tie), next_town, path + [next_town]))

    # The frontier ran empty: every possible route is blocked or leads nowhere
    return None, f"No open route exists from {start} to {goal}."

In [11]:
# ------------------------------------------------------------
# STEP 7b: QUERY FUNCTIONS - the questions a user can ask the agent
# Each one calls the KB (through the Step 4 helpers or ucs) and PRINTS
# a readable answer.
# ------------------------------------------------------------

def query_best_route(start=START, goal=GOAL, explain=True):
    """Q1: What is the best route from start to goal? (defaults to our trip)"""
    path, result = ucs(start, goal)
    if path is None:
        print(f"Best route {start} -> {goal}: {result}")
        return None
    print(f"Best route {normalise(start)} -> {normalise(goal)}: {' -> '.join(path)}")
    print(f"Total effective cost: {result} km-equivalent")
    if explain and len(path) > 1:                      # show every leg of the trip
        for a, b in zip(path, path[1:]):
            km = kb["roads"][a][b]
            hazards = kb["hazards"].get((a, b), [])
            extra = f"  + hazards {hazards} (+{hazard_penalty(a, b)})" if hazards else ""
            print(f"   {a:10} -> {b:10} {km:4} km{extra}")
    return path


def query_reachable(start, goal):
    """Q2: Can the trailer get from start to goal at all (ignoring cost)?"""
    path, result = ucs(start, goal)
    answer = path is not None
    print(f"Can we reach {normalise(goal)} from {normalise(start)}? {'YES' if answer else 'NO'}"
          + ("" if answer else f"  ({result})"))
    return answer


def query_cost(start, goal):
    """Q3: What is the cheapest cost between two towns?"""
    path, result = ucs(start, goal)
    if path is None:
        print(f"Cost {normalise(start)} -> {normalise(goal)}: not available. {result}")
        return None
    print(f"Cheapest cost {normalise(start)} -> {normalise(goal)}: {result} km-equivalent via {' -> '.join(path)}")
    return result


def query_road(a, b):
    """Q4: What do we know about one specific road?"""
    a, b = normalise(a), normalise(b)
    if not road_exists(a, b):
        print(f"There is no direct road {a} -> {b} in the KB.")
        return None
    km = kb["roads"][a][b]
    status = kb["status"][(a, b)]
    hazards = kb["hazards"].get((a, b), [])
    print(f"Road {a} -> {b}: {km} km, {status.upper()}, hazards: {hazards if hazards else 'none'}, "
          f"effective cost: {km + hazard_penalty(a, b) if status == 'open' else 'n/a (closed)'}")
    return km


def query_reachable_from(town):
    """Q5: Which towns can we get to from this town? (breadth-first search over OPEN roads)"""
    town = normalise(town)
    if not town_exists(town):
        print(f"I don't know a town called '{town}'.")
        return None
    seen, queue = {town}, [town]
    while queue:
        current = queue.pop(0)
        for nxt, _ in get_neighbors(current):
            if nxt not in seen:
                seen.add(nxt)
                queue.append(nxt)
    seen.discard(town)
    print(f"From {town} we can reach {len(seen)} town(s): {sorted(seen) if seen else 'none (dead end)'}")
    return seen


def query_active_problems():
    """Q6: List every closed road and every hazard currently in the KB."""
    closed = [f"{a} -> {b}" for (a, b), s in kb["status"].items() if s == "closed"]
    hazards = {f"{a} -> {b}": h for (a, b), h in kb["hazards"].items() if h}
    print("Closed roads:", closed if closed else "none")
    print("Roads with hazards:", hazards if hazards else "none")
    return closed, hazards

In [12]:
# ------------------------------------------------------------
# STEP 7c: USER INTERACTION - a text menu
# input() pauses the notebook, so run_menu() is NOT called automatically
# (that would stop "Run all"). Remove the # on the last line to try it live.
# ------------------------------------------------------------

MENU = """
===== AUTONOMOUS TRAILER AGENT =====
 1. Best route (Nairobi -> Kisumu)
 2. Best route between two towns
 3. Can I reach a town from another?
 4. Cost between two towns
 5. Info about one road
 6. Which towns can I reach from here?
 7. Show closed roads and hazards
 8. Close or open a road
 9. Report a hazard on a road
 0. Quit
"""

def run_menu(read=input):
    """Loop: show the menu, read a choice, answer it. `read` is the function used
    to get the user's typing. By default it is input(); in Step 8 we pass a fake
    one so we can demonstrate the menu without typing."""
    while True:
        print(MENU)
        choice = read("Choose an option: ").strip()
        if choice == "1":
            query_best_route()
        elif choice == "2":
            query_best_route(read("From: "), read("To: "))
        elif choice == "3":
            query_reachable(read("From: "), read("To: "))
        elif choice == "4":
            query_cost(read("From: "), read("To: "))
        elif choice == "5":
            query_road(read("From: "), read("To: "))
        elif choice == "6":
            query_reachable_from(read("Town: "))
        elif choice == "7":
            query_active_problems()
        elif choice == "8":
            set_status(read("From: "), read("To: "), read("New status (open/closed): "))
        elif choice == "9":
            add_hazard(read("From: "), read("To: "), read("Hazard: "))
        elif choice == "0":
            print("Goodbye.")
            break
        else:
            print(f"'{choice}' is not a valid option. Please choose 0-9.")

# run_menu()      # <- remove the # to use the menu yourself

**Step 8: Querying the knowledge base**

We ask the agent a range of questions, then **change the world** (close roads, clear hazards) and ask again. If the answers change correctly, the agent is genuinely reasoning from the KB and not from a fixed answer.

In [13]:
# ------------------------------------------------------------
# STEP 8: QUERYING THE KNOWLEDGE BASE
# Section 1 uses TODAY's conditions from Step 6.
# Section 2 changes the world and asks again, to prove the agent reasons from the KB.
# ------------------------------------------------------------
restore_kb(TODAY_KB)

print("########## 8.1 QUESTIONS ABOUT TODAY'S KB ##########")
print("\n[Q1] Best route for the trip")
query_best_route()

print("\n[Q2] Reachability")
query_reachable("Nairobi", "Kisumu")
query_reachable("Nairobi", "Migori")

print("\n[Q3] Cost between two towns")
query_cost("Nairobi", "Nakuru")
query_cost("Nairobi", "Kisii")

print("\n[Q4] One road")
query_road("Narok", "Bomet")
query_road("Nairobi", "Kisumu")          # no such direct road

print("\n[Q5] Towns reachable from Narok")
query_reachable_from("Narok")

print("\n[Q6] Problems currently known")
query_active_problems()

# ------------------------------------------------------------
print("\n\n########## 8.2 WHAT HAPPENS WHEN THE WORLD CHANGES? ##########")

print("\n--- Baseline (clean KB, no hazards) ---")
restore_kb(BASE_KB)
baseline = query_best_route(explain=False)

print("\n--- Today's conditions (fog + roadworks + flooding) ---")
restore_kb(TODAY_KB)
query_best_route(explain=False)
print("Note: the route is the same but costs more, because the hazards add penalties.")

print("\n--- Scenario A: Naivasha -> Gilgil is CLOSED (the 354 km corridor is blocked) ---")
set_status("Naivasha", "Gilgil", "closed")
query_best_route()                       # the agent switches to the Narok - Bomet - Kisii corridor

print("\n--- Scenario B: the flooding on Narok -> Bomet is cleared ---")
clear_hazards("Narok", "Bomet")
query_best_route()                       # same route as A, but 40 km cheaper

print("\n--- Scenario C: ALSO close Kisii -> Kisumu ---")
set_status("Kisii", "Kisumu", "closed")
query_best_route()                       # must now go Kisii -> Homa Bay -> Kisumu

print("\n--- Scenario D: ALSO close Narok -> Bomet (the whole southern corridor is gone) ---")
set_status("Narok", "Bomet", "closed")
query_best_route(explain=False)          # a third corridor exists: through Ruiru, Thika and Nyeri

print("\n--- Scenario E: ALSO close Nakuru -> Kericho (no way left to Kisumu) ---")
set_status("Nakuru", "Kericho", "closed")
query_best_route()                       # the agent should say there is no route
query_reachable("Nairobi", "Kisumu")

# ------------------------------------------------------------
print("\n\n########## 8.3 THE MENU, DRIVEN BY A SCRIPTED USER ##########")
restore_kb(TODAY_KB)
script = iter(["1", "2", "Nairobi", "Kisii", "8", "nairobi", "naivasha", "closed",
               "1", "7", "x", "0"])           # the 'typed' answers, in order
def fake_input(prompt):
    answer = next(script)
    print(f"{prompt}{answer}")                # show what the 'user' typed
    return answer
run_menu(read=fake_input)

restore_kb(TODAY_KB)                          # leave the KB clean for Step 9

########## 8.1 QUESTIONS ABOUT TODAY'S KB ##########

[Q1] Best route for the trip
Best route Nairobi -> Kisumu: Nairobi -> Naivasha -> Gilgil -> Nakuru -> Kericho -> Kisumu
Total effective cost: 384 km-equivalent
   Nairobi    -> Naivasha     91 km
   Naivasha   -> Gilgil       30 km  + hazards ['roadworks'] (+20)
   Gilgil     -> Nakuru       42 km
   Nakuru     -> Kericho     107 km  + hazards ['fog'] (+10)
   Kericho    -> Kisumu       84 km

[Q2] Reachability
Can we reach Kisumu from Nairobi? YES
Can we reach Migori from Nairobi? YES

[Q3] Cost between two towns
Cheapest cost Nairobi -> Nakuru: 183 km-equivalent via Nairobi -> Naivasha -> Gilgil -> Nakuru
Cheapest cost Nairobi -> Kisii: 346 km-equivalent via Nairobi -> Narok -> Bomet -> Kisii

[Q4] One road
Road Narok -> Bomet: 81 km, OPEN, hazards: ['flooding'], effective cost: 121
There is no direct road Nairobi -> Kisumu in the KB.

[Q5] Towns reachable from Narok
From Narok we can reach 5 town(s): ['Bomet', 'Homa Bay', 'Kisii'

**Step 9: Debugging the Knowledge Base** `[2 marks]`

We test faulty and edge-case inputs: unknown towns, dead ends, blocked roads, a cycle, a missing distance, a deleted hazard rule and wrong data types. **Step 9a** runs the tests on the existing code and shows what breaks. **Step 9b** fixes it. The fixes in 9b deliberately **replace** the earlier versions of `normalise`, `hazard_penalty`, `get_neighbors`, `set_status` and `add_hazard` from Steps 4 and 5. **Step 9c** runs the same tests again to prove the fixes work.

In [14]:
# ------------------------------------------------------------
# STEP 9a: DEBUGGING - throw faulty and edge-case inputs at the agent
# `try_it` runs one test and reports either the answer or a CRASH,
# so one failure does not stop the remaining tests.
# ------------------------------------------------------------
def try_it(label, func, *args):
    print(f"\nTEST: {label}")
    try:
        return func(*args)
    except Exception as e:                                   # catch ANY crash
        print(f"   *** CRASH: {type(e).__name__}: {e}")

def run_edge_tests():
    restore_kb(TODAY_KB)
    # 1. Unknown towns
    try_it("unknown destination", query_best_route, "Nairobi", "Mombasa")
    try_it("unknown start", query_best_route, "Eldoret", "Kisumu")

    # 2. Messy typing (should still work)
    try_it("lower case + extra spaces", query_cost, "  nairobi ", "KISUMU")

    # 3. Dead end: Makindu has no outgoing roads
    try_it("start at a dead end", query_best_route, "Makindu", "Kisumu")
    try_it("go TO a dead end (valid, it just ends there)", query_cost, "Nairobi", "Makindu")

    # 4. Trivial question
    try_it("start equals goal", query_best_route, "Kisumu", "Kisumu")

    # 5. Blocked roads: the goal becomes unreachable
    def block_kisumu():
        for a, b in [("Kericho", "Kisumu"), ("Kisii", "Kisumu"), ("Homa Bay", "Kisumu")]:
            set_status(a, b, "closed")
    try_it("close every road into Kisumu", block_kisumu)
    try_it("goal unreachable", query_best_route, "Nairobi", "Kisumu")
    restore_kb(TODAY_KB)

    # 6. A cycle in the graph (UCS must not loop forever)
    add_road("Nakuru", "Naivasha", 60)                          # Naivasha -> Gilgil -> Nakuru -> Naivasha
    try_it("cycle in the network", query_best_route, "Nairobi", "Kisumu", False)
    restore_kb(TODAY_KB)

    # 7. Missing distance: a road whose distance is None (corrupted fact)
    kb["roads"]["Nairobi"]["Naivasha"] = None
    try_it("road with a missing distance", query_best_route, "Nairobi", "Kisumu")
    restore_kb(TODAY_KB)

    # 8. A hazard whose type is no longer known
    del kb["rules"]["hazard_penalty_km"]["fog"]                 # forget fog, but Nakuru -> Kericho still lists it
    try_it("hazard type that was deleted from the rules", query_best_route, "Nairobi", "Kisumu")
    restore_kb(TODAY_KB)

    # 9. Wrong data types typed by a user
    try_it("town given as None", query_best_route, None, "Kisumu")
    try_it("town given as a number", query_best_route, 42, "Kisumu")
    try_it("status given as None", set_status, "Nairobi", "Ruiru", None)
    try_it("hazard given as None", add_hazard, "Nairobi", "Ruiru", None)
    try_it("negative distance", add_road, "Nairobi", "Kisumu", -5)
    try_it("missing distance in add_road", add_road, "Nairobi", "Kisumu", None)
    restore_kb(TODAY_KB)


run_edge_tests()


TEST: unknown destination
Best route Nairobi -> Mombasa: I don't know a town called 'Mombasa'.

TEST: unknown start
Best route Eldoret -> Kisumu: I don't know a town called 'Eldoret'.

TEST: lower case + extra spaces
Cheapest cost Nairobi -> Kisumu: 384 km-equivalent via Nairobi -> Naivasha -> Gilgil -> Nakuru -> Kericho -> Kisumu

TEST: start at a dead end
Best route Makindu -> Kisumu: No open route exists from Makindu to Kisumu.

TEST: go TO a dead end (valid, it just ends there)
Cheapest cost Nairobi -> Makindu: 179 km-equivalent via Nairobi -> Athi River -> Makindu

TEST: start equals goal
Best route Kisumu -> Kisumu: Kisumu
Total effective cost: 0 km-equivalent

TEST: close every road into Kisumu
Road Kericho -> Kisumu is now CLOSED
Road Kisii -> Kisumu is now CLOSED
Road Homa Bay -> Kisumu is now CLOSED

TEST: goal unreachable
Best route Nairobi -> Kisumu: No open route exists from Nairobi to Kisumu.
Added road: Nakuru -> Naivasha (60 km, open)

TEST: cycle in the network
Best r

In [15]:
# ------------------------------------------------------------
# STEP 9b: FIXES - repair the functions that crashed in Step 9a
# These definitions REPLACE the earlier ones with the same name.
# Because every other function looks the name up when it runs, they
# all pick up the fix automatically.
# ------------------------------------------------------------

# FIX 1: normalise() crashed on None and numbers (they have no .strip()).
# Now anything that is not text becomes an empty string, which is
# "an unknown town", so the agent answers politely instead of crashing.
def normalise(name):
    """Clean up a town name. Non-text input becomes '' (an unknown town)."""
    if not isinstance(name, str):
        return ""
    return name.strip().title()


# FIX 2: hazard_penalty() crashed when a road listed a hazard that has no rule.
# Now an unknown hazard is given a cautious DEFAULT penalty (and a warning),
# which is safer than ignoring it or stopping.
DEFAULT_PENALTY = 20

def hazard_penalty(a, b):
    """Total penalty of the hazards on road a -> b (unknown hazards use DEFAULT_PENALTY)."""
    a, b = normalise(a), normalise(b)
    penalties = kb["rules"]["hazard_penalty_km"]
    total = 0
    for h in kb["hazards"].get((a, b), []):
        if h not in penalties:
            print(f"   (warning: no penalty rule for '{h}' on {a} -> {b}; using {DEFAULT_PENALTY})")
        total += penalties.get(h, DEFAULT_PENALTY)
    return total


# FIX 3: get_neighbors() crashed when a road had a missing or invalid distance.
# Now such a road is SKIPPED with a warning: we cannot cost it, so we do not drive it.
def get_neighbors(town):
    """List of (next_town, effective_cost) for every open road with a valid distance."""
    town = normalise(town)
    neighbors = []
    for next_town, km in kb["roads"].get(town, {}).items():
        if not is_reachable_road(town, next_town):
            continue                                         # closed roads are left out
        if not isinstance(km, (int, float)) or km <= 0:
            print(f"   (warning: road {town} -> {next_town} has no valid distance, so it is ignored)")
            continue
        neighbors.append((next_town, km + hazard_penalty(town, next_town)))
    return neighbors


# FIX 4: set_status() and add_hazard() crashed on non-text input (None.strip()).
def set_status(a, b, status):
    """Open or close a road. Only the values in kb['rules']['valid_status'] are allowed."""
    a, b = normalise(a), normalise(b)
    if not isinstance(status, str):
        print("Rejected: status must be text ('open' or 'closed').")
        return False
    status = status.strip().lower()
    if not road_exists(a, b):
        print(f"Rejected: no road {a} -> {b}.")
        return False
    if status not in kb["rules"]["valid_status"]:
        print(f"Rejected: status must be one of {kb['rules']['valid_status']}.")
        return False
    kb["status"][(a, b)] = status
    print(f"Road {a} -> {b} is now {status.upper()}")
    return True


def add_hazard(a, b, hazard):
    """Record that a known hazard is present on the road a -> b."""
    a, b = normalise(a), normalise(b)
    if not isinstance(hazard, str):
        print("Rejected: hazard must be text, e.g. 'fog'.")
        return False
    hazard = hazard.strip().lower()
    if not road_exists(a, b):
        print(f"Rejected: no road {a} -> {b}.")
        return False
    if hazard not in kb["rules"]["hazard_penalty_km"]:
        print(f"Rejected: unknown hazard '{hazard}'. Known: {list(kb['rules']['hazard_penalty_km'])}")
        return False
    current = kb["hazards"].setdefault((a, b), [])
    if hazard in current:
        print(f"Rejected: '{hazard}' is already recorded on {a} -> {b}.")
        return False
    current.append(hazard)
    print(f"Added hazard '{hazard}' on {a} -> {b}")
    return True

print("Fixes 1-4 applied.")

Fixes 1-4 applied.


In [16]:
# ------------------------------------------------------------
# STEP 9c: RE-RUN THE SAME TESTS AFTER THE FIXES
# Compare with Step 9a: every *** CRASH line should now be a polite message.
# ------------------------------------------------------------
run_edge_tests()
restore_kb(TODAY_KB)

# Final regression check: the original behaviour must be unchanged
restore_kb(BASE_KB)
assert route_cost(KNOWN_BEST_ROUTE) == 354
assert ucs("Nairobi", "Kisumu") == (KNOWN_BEST_ROUTE, 354)
restore_kb(TODAY_KB)
print("\nRegression check passed: clean KB still gives the 354 km route.")


TEST: unknown destination
Best route Nairobi -> Mombasa: I don't know a town called 'Mombasa'.

TEST: unknown start
Best route Eldoret -> Kisumu: I don't know a town called 'Eldoret'.

TEST: lower case + extra spaces
Cheapest cost Nairobi -> Kisumu: 384 km-equivalent via Nairobi -> Naivasha -> Gilgil -> Nakuru -> Kericho -> Kisumu

TEST: start at a dead end
Best route Makindu -> Kisumu: No open route exists from Makindu to Kisumu.

TEST: go TO a dead end (valid, it just ends there)
Cheapest cost Nairobi -> Makindu: 179 km-equivalent via Nairobi -> Athi River -> Makindu

TEST: start equals goal
Best route Kisumu -> Kisumu: Kisumu
Total effective cost: 0 km-equivalent

TEST: close every road into Kisumu
Road Kericho -> Kisumu is now CLOSED
Road Kisii -> Kisumu is now CLOSED
Road Homa Bay -> Kisumu is now CLOSED

TEST: goal unreachable
Best route Nairobi -> Kisumu: No open route exists from Nairobi to Kisumu.
Added road: Nakuru -> Naivasha (60 km, open)

TEST: cycle in the network
Best r

**Step 10: Final Reflection** `[3 marks]`

**What we built.** A knowledge-based agent for the Nairobi to Kisumu trailer. The KB is one dictionary (towns, roads, status, hazards, rules). General knowledge (the 21 roads and the hazard penalty rules) is kept separate from the facts of one trip (today's hazards), and the UCS from practical 3 now queries the KB through `get_neighbors` instead of a fixed graph. When we closed roads or cleared hazards, the route changed correctly (354 km, 384 km with today's conditions, 460 km via Narok, 466 km via Homa Bay, 532 km via Ruiru, and finally "no route"), which shows the agent reasons from its knowledge and not from a stored answer.

**Challenges**
1. *Keeping the KB and the search separate.* The search must never read `kb["roads"]` directly, otherwise a closed road or a hazard would be ignored. Routing everything through `get_neighbors` solved this, but it needed us to agree the interface first.
2. *Testing without damaging the KB.* Each test changed the KB and affected the next one. We solved this with snapshots (`BASE_KB`, `TODAY_KB`) and `restore_kb`.
3. *Bad input.* Step 9 showed that the first version crashed on `None`, numbers, a missing distance and a hazard with no rule. A silent problem was also found while testing: a default of `start or START` would have quietly replaced a missing town with Nairobi, giving a confident but wrong answer.
4. *Choosing hazard penalties.* The penalties (fog = 10, flooding = 40, and so on) are our own judgement, not measured data, so a different choice could change which route wins.
5. *Interaction in a notebook.* `input()` pauses "Run all", so the menu is only called with scripted input in the demo.
6. *Designing the KB structure.* Our practical 3 graph was a list of tuples, which made it awkward to change one road at runtime. We switched to nested dictionaries (`from -> {to: km}`) and kept road status and hazards in separate sections, so closing a road or adding a hazard never touches the original 21 distances. Each update function also needed input checks (unknown town, bad status, negative distance) so a wrong fact could not corrupt the KB.

**Recommended improvements**
- Fill hazards from live data (a weather API) instead of typing them, with a saved copy so the notebook still runs offline.
- Add time-based costs (speed limits, traffic, fuel) and not just distance and penalties.
- Make the roads two-way, since our graph is directed and has no return roads.
- Use real coordinates and a heuristic so the search can be upgraded to A*.
- Store the KB in a file so new facts survive between sessions.
- Add automatic validation that rejects impossible distances before they enter the KB.

## Evidence of group work

![Our team](team_photo.jpeg)